# Common source-validation sharpness proxy
Written only; no cells executed during creation. See README.md for run order.

In [ ]:
def sharpness(model, x, y, radius=0.05):
    """One global gradient-ascent perturbation; restore every parameter exactly."""
    model.eval()
    parameters = [p for p in model.parameters() if p.requires_grad]
    clean = nn.functional.cross_entropy(model(x), y)
    grads = torch.autograd.grad(clean, parameters)
    norm = torch.stack([g.norm(2) for g in grads]).norm(2)
    if not torch.isfinite(norm) or not torch.isfinite(clean):
        raise RuntimeError('Non-finite sharpness gradient/loss.')
    originals = [p.detach().clone() for p in parameters]
    try:
        with torch.no_grad():
            for p, g in zip(parameters, grads):
                p.add_(radius * g / norm.clamp_min(1e-12))
            perturbed = nn.functional.cross_entropy(model(x), y)
    finally:
        with torch.no_grad():
            for p, original in zip(parameters, originals):
                p.copy_(original)
    if not torch.isfinite(perturbed):
        raise RuntimeError('Non-finite perturbed sharpness loss.')
    return dict(sharpness=float(perturbed-clean.detach()), sharpness_clean_loss=float(clean.detach()),
                sharpness_perturbed_loss=float(perturbed), sharpness_gradient_norm=float(norm))


